# Notebook 15: Quantum Kernel Methods

**Series: Quantum Computing from Ground Up**

---

## Learning Objectives

1. Review classical kernel methods and Support Vector Machines (SVMs)
2. Understand the quantum kernel trick and quantum feature maps
3. Derive and compute quantum kernel matrices
4. Analyze when quantum kernels can offer advantages
5. Explore projected quantum kernels
6. Implement a quantum kernel SVM and compare it with classical SVM on the Iris dataset

## 1. Classical Kernel Methods Review

### 1.1 The Kernel Trick

Many machine learning algorithms (SVM, kernel PCA, Gaussian processes) rely on **dot products** between data points. The kernel trick allows us to compute these dot products in a high-dimensional feature space without ever explicitly computing the features.

Given a feature map $\phi: \mathbb{R}^d \to \mathcal{F}$, the **kernel function** is:

$$k(\mathbf{x}, \mathbf{x}') = \langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle_{\mathcal{F}}$$

### 1.2 Support Vector Machines

The SVM finds the maximum-margin hyperplane in feature space. The decision function is:

$$f(\mathbf{x}) = \text{sign}\left(\sum_{i=1}^{N} \alpha_i y_i k(\mathbf{x}_i, \mathbf{x}) + b\right)$$

where the $\alpha_i$ are found by solving the **dual optimization** problem:

$$\max_{\boldsymbol{\alpha}} \sum_i \alpha_i - \frac{1}{2} \sum_{i,j} \alpha_i \alpha_j y_i y_j k(\mathbf{x}_i, \mathbf{x}_j)$$

subject to $0 \leq \alpha_i \leq C$ and $\sum_i \alpha_i y_i = 0$.

### 1.3 Common Classical Kernels

| Kernel | Formula | Feature Space Dimension |
|--------|---------|------------------------|
| Linear | $k(\mathbf{x}, \mathbf{x}') = \mathbf{x}^T \mathbf{x}'$ | $d$ |
| Polynomial | $k(\mathbf{x}, \mathbf{x}') = (\mathbf{x}^T \mathbf{x}' + c)^p$ | $\binom{d+p}{p}$ |
| RBF (Gaussian) | $k(\mathbf{x}, \mathbf{x}') = \exp(-\gamma\|\mathbf{x} - \mathbf{x}'\|^2)$ | $\infty$ |

### 1.4 Mercer's Theorem

A function $k: \mathcal{X} \times \mathcal{X} \to \mathbb{R}$ is a valid kernel if and only if the kernel matrix $K_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$ is **positive semi-definite** for any finite set of data points. Equivalently, there exists a feature map $\phi$ such that $k(\mathbf{x}, \mathbf{x}') = \langle \phi(\mathbf{x}), \phi(\mathbf{x}') \rangle$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import load_iris, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report

# Demonstrate the kernel trick with a simple 2D example
np.random.seed(42)

# Generate XOR-like data (not linearly separable)
n_samples = 200
X_xor = np.random.randn(n_samples, 2)
y_xor = np.array([1 if x[0] * x[1] > 0 else -1 for x in X_xor])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original data
ax = axes[0]
ax.scatter(X_xor[y_xor == 1, 0], X_xor[y_xor == 1, 1], c='blue', marker='o', label='Class +1', alpha=0.6)
ax.scatter(X_xor[y_xor == -1, 0], X_xor[y_xor == -1, 1], c='red', marker='x', label='Class -1', alpha=0.6)
ax.set_title('Original Data (not linearly separable)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)

# Linear SVM
svm_linear = SVC(kernel='linear', C=1.0)
svm_linear.fit(X_xor, y_xor)
ax = axes[1]
xx, yy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
Z_lin = svm_linear.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z_lin, levels=20, cmap='RdBu', alpha=0.5)
ax.contour(xx, yy, Z_lin, levels=[0], colors='black', linewidths=2)
ax.scatter(X_xor[y_xor == 1, 0], X_xor[y_xor == 1, 1], c='blue', marker='o', alpha=0.6)
ax.scatter(X_xor[y_xor == -1, 0], X_xor[y_xor == -1, 1], c='red', marker='x', alpha=0.6)
ax.set_title(f'Linear SVM (acc={svm_linear.score(X_xor, y_xor):.1%})', fontsize=12)
ax.grid(True, alpha=0.3)

# RBF SVM
svm_rbf = SVC(kernel='rbf', gamma=1.0, C=1.0)
svm_rbf.fit(X_xor, y_xor)
ax = axes[2]
Z_rbf = svm_rbf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z_rbf, levels=20, cmap='RdBu', alpha=0.5)
ax.contour(xx, yy, Z_rbf, levels=[0], colors='black', linewidths=2)
ax.scatter(X_xor[y_xor == 1, 0], X_xor[y_xor == 1, 1], c='blue', marker='o', alpha=0.6)
ax.scatter(X_xor[y_xor == -1, 0], X_xor[y_xor == -1, 1], c='red', marker='x', alpha=0.6)
ax.set_title(f'RBF SVM (acc={svm_rbf.score(X_xor, y_xor):.1%})', fontsize=12)
ax.grid(True, alpha=0.3)

plt.suptitle('Classical Kernel Methods: The Power of Feature Maps', fontsize=14)
plt.tight_layout()
plt.show()

## 2. The Quantum Kernel

### 2.1 Quantum Feature Map

A **quantum feature map** encodes classical data $\mathbf{x} \in \mathbb{R}^d$ into an $n$-qubit quantum state:

$$|\phi(\mathbf{x})\rangle = U_{\phi}(\mathbf{x})|0\rangle^{\otimes n}$$

The feature space is the $2^n$-dimensional Hilbert space, and the features are the amplitudes of the quantum state.

### 2.2 Quantum Kernel Definition

The **quantum kernel** (or **fidelity kernel**) is:

$$\kappa(\mathbf{x}, \mathbf{x}') = |\langle \phi(\mathbf{x}) | \phi(\mathbf{x}') \rangle|^2 = |\langle 0^n | U_{\phi}^\dagger(\mathbf{x}) U_{\phi}(\mathbf{x}') | 0^n \rangle|^2$$

This is the **transition probability** between the two encoded states.

### 2.3 Computing the Quantum Kernel

On a quantum computer, $\kappa(\mathbf{x}, \mathbf{x}')$ is estimated by:

1. Prepare $|\phi(\mathbf{x}')\rangle = U_\phi(\mathbf{x}')|0^n\rangle$
2. Apply $U_\phi^\dagger(\mathbf{x})$
3. Measure all qubits
4. Repeat $M$ times
5. $\hat{\kappa} = \frac{\text{number of } |0^n\rangle \text{ outcomes}}{M}$

### 2.4 Properties

The quantum kernel is a valid kernel (positive semi-definite) because:

$$\kappa(\mathbf{x}, \mathbf{x}') = \text{Tr}(\rho(\mathbf{x}) \cdot \rho(\mathbf{x}'))$$

where $\rho(\mathbf{x}) = |\phi(\mathbf{x})\rangle\langle\phi(\mathbf{x})|$. This is the **Hilbert-Schmidt inner product** of density matrices, which is always PSD by construction.

In [ ]:
import pennylane as qml

# Define quantum feature maps of varying complexity

n_qubits = 2
dev = qml.device('default.qubit', wires=n_qubits)

def feature_map_simple(x, wires):
    """Simple angle encoding."""
    for i, w in enumerate(wires):
        qml.RY(x[i], wires=w)

def feature_map_iqp(x, wires):
    """IQP-style feature map with ZZ entanglement."""
    n = len(wires)
    # First layer
    for i, w in enumerate(wires):
        qml.Hadamard(wires=w)
        qml.RZ(x[i], wires=w)
    # ZZ interactions
    for i in range(n):
        for j in range(i + 1, n):
            qml.CNOT(wires=[wires[i], wires[j]])
            qml.RZ(x[i] * x[j], wires=wires[j])
            qml.CNOT(wires=[wires[i], wires[j]])
    # Second layer (repeat for depth)
    for i, w in enumerate(wires):
        qml.Hadamard(wires=w)
        qml.RZ(x[i], wires=w)
    for i in range(n):
        for j in range(i + 1, n):
            qml.CNOT(wires=[wires[i], wires[j]])
            qml.RZ(x[i] * x[j], wires=wires[j])
            qml.CNOT(wires=[wires[i], wires[j]])

def feature_map_hardware_efficient(x, wires):
    """Hardware-efficient feature map."""
    n = len(wires)
    for layer in range(2):
        for i, w in enumerate(wires):
            qml.RY(x[i], wires=w)
            qml.RZ(x[i] ** 2, wires=w)
        for i in range(n - 1):
            qml.CNOT(wires=[wires[i], wires[i + 1]])
        qml.CNOT(wires=[wires[-1], wires[0]])  # Circular entanglement

print("Defined three quantum feature maps:")
print("  1. Simple angle encoding (no entanglement)")
print("  2. IQP feature map (ZZ interactions)")
print("  3. Hardware-efficient feature map")

In [ ]:
# Implement quantum kernel computation

def make_kernel_circuit(feature_map_fn, n_qubits):
    """Create a quantum kernel circuit for a given feature map."""
    dev_k = qml.device('default.qubit', wires=n_qubits)
    
    @qml.qnode(dev_k)
    def kernel_circuit(x1, x2):
        # Apply U_phi(x2)
        feature_map_fn(x2, wires=range(n_qubits))
        # Apply U_phi^dagger(x1)
        qml.adjoint(feature_map_fn)(x1, wires=range(n_qubits))
        # Measure probability of |0...0>
        return qml.probs(wires=range(n_qubits))
    
    def kernel(x1, x2):
        return kernel_circuit(x1, x2)[0]
    
    return kernel

def compute_kernel_matrix(kernel_fn, X1, X2=None):
    """Compute the kernel matrix K[i,j] = kernel(X1[i], X2[j])."""
    if X2 is None:
        X2 = X1
    n1, n2 = len(X1), len(X2)
    K = np.zeros((n1, n2))
    for i in range(n1):
        for j in range(n2):
            K[i, j] = kernel_fn(X1[i], X2[j])
    return K

# Test kernel computation
kernel_simple = make_kernel_circuit(feature_map_simple, n_qubits)
kernel_iqp = make_kernel_circuit(feature_map_iqp, n_qubits)
kernel_he = make_kernel_circuit(feature_map_hardware_efficient, n_qubits)

# Test with sample points
x1 = np.array([0.5, 1.0])
x2 = np.array([0.5, 1.0])
x3 = np.array([2.0, 0.3])

print("Kernel values κ(x, x'):")
print(f"  Simple:  κ(x1,x1)={kernel_simple(x1,x1):.4f}, κ(x1,x3)={kernel_simple(x1,x3):.4f}")
print(f"  IQP:     κ(x1,x1)={kernel_iqp(x1,x1):.4f}, κ(x1,x3)={kernel_iqp(x1,x3):.4f}")
print(f"  HW-eff:  κ(x1,x1)={kernel_he(x1,x1):.4f}, κ(x1,x3)={kernel_he(x1,x3):.4f}")
print("\nNote: κ(x,x) = 1 always (identity property).")

## 3. Quantum Kernel Mathematics

### 3.1 Kernel as Fourier Transform

For angle-encoding feature maps, the quantum kernel can be expressed as:

$$\kappa(\mathbf{x}, \mathbf{x}') = \sum_{\omega \in \Omega} c_\omega \, e^{i\omega \cdot (\mathbf{x} - \mathbf{x}')}$$

where $\Omega$ is the set of frequencies determined by the encoding gates' eigenvalue spectra.

### 3.2 Bandwidth and Expressibility

The **bandwidth** of the quantum kernel — determined by $|\Omega|$ — controls its expressibility:

- **Low bandwidth** (few frequencies): smooth kernel, underfitting risk
- **High bandwidth** (many frequencies): complex kernel, overfitting risk

### 3.3 Kernel Alignment

A kernel's quality for a specific task is measured by **kernel-target alignment**:

$$A(K, \mathbf{y}) = \frac{\langle K, \mathbf{y}\mathbf{y}^T \rangle_F}{\|K\|_F \cdot \|\mathbf{y}\mathbf{y}^T\|_F}$$

where $\langle A, B \rangle_F = \text{Tr}(A^T B)$ is the Frobenius inner product.

Higher alignment means the kernel better captures the structure of the labels.

### 3.4 Geometric Difference

Huang et al. (2021) introduced the **geometric difference** to quantify potential quantum advantage:

$$g(K_Q, K_C) = \sqrt{\|\sqrt{K_C} K_Q^{-1} \sqrt{K_C}\|}$$

If $g \gg 1$, the quantum kernel has features that are "geometrically different" from any efficient classical kernel, suggesting potential advantage.

In [ ]:
# Visualize quantum kernels for 1D data

x_1d = np.linspace(0, 2*np.pi, 50)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
kernels_info = [
    (kernel_simple, 'Simple (Angle)'),
    (kernel_iqp, 'IQP Feature Map'),
    (kernel_he, 'Hardware Efficient'),
]

for idx, (kern, name) in enumerate(kernels_info):
    ax = axes[idx]
    # Fix x1 = pi, vary x2
    x_ref = np.array([np.pi, np.pi/2])
    k_vals = []
    for xi in x_1d:
        x_var = np.array([xi, np.pi/2])
        k_vals.append(kern(x_ref, x_var))
    
    ax.plot(x_1d, k_vals, 'b-', linewidth=2)
    ax.axvline(x=np.pi, color='r', linestyle='--', alpha=0.5, label='x_ref = π')
    ax.set_xlabel('x₁', fontsize=12)
    ax.set_ylabel('κ(x_ref, x)', fontsize=12)
    ax.set_title(f'{name} Kernel', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.05, 1.05)

plt.suptitle('Quantum Kernel Profiles (fixing x₂ = π/2, varying x₁)', fontsize=14)
plt.tight_layout()
plt.show()

print("Different feature maps create different kernel shapes:")
print("  - Simple: smooth, broad kernel")
print("  - IQP: sharper, more complex structure (higher bandwidth)")
print("  - HW-efficient: intermediate complexity")

In [ ]:
# Compute kernel-target alignment for different feature maps

def kernel_target_alignment(K, y):
    """Compute kernel-target alignment."""
    yy = np.outer(y, y)
    alignment = np.sum(K * yy) / (np.linalg.norm(K, 'fro') * np.linalg.norm(yy, 'fro'))
    return alignment

# Generate a simple classification dataset
np.random.seed(42)
X_align, y_align = make_moons(n_samples=40, noise=0.15, random_state=42)
y_align = 2 * y_align - 1

# Scale to [0, pi]
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_align_scaled = scaler.fit_transform(X_align)

# Compute kernel matrices
print("Computing kernel matrices (this may take a moment)...")
K_simple = compute_kernel_matrix(kernel_simple, X_align_scaled)
K_iqp = compute_kernel_matrix(kernel_iqp, X_align_scaled)
K_he = compute_kernel_matrix(kernel_he, X_align_scaled)

# Classical RBF kernel for comparison
from sklearn.metrics.pairwise import rbf_kernel
K_rbf = rbf_kernel(X_align_scaled, gamma=1.0)

# Compute alignments
a_simple = kernel_target_alignment(K_simple, y_align)
a_iqp = kernel_target_alignment(K_iqp, y_align)
a_he = kernel_target_alignment(K_he, y_align)
a_rbf = kernel_target_alignment(K_rbf, y_align)

print(f"\nKernel-Target Alignment (higher = better for this task):")
print(f"  Simple quantum kernel:  {a_simple:.4f}")
print(f"  IQP quantum kernel:     {a_iqp:.4f}")
print(f"  HW-efficient kernel:    {a_he:.4f}")
print(f"  Classical RBF kernel:   {a_rbf:.4f}")

# Visualize kernel matrices
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, K, name in zip(axes, [K_simple, K_iqp, K_he, K_rbf], 
                        ['Simple QK', 'IQP QK', 'HW-eff QK', 'Classical RBF']):
    im = ax.imshow(K, cmap='viridis', vmin=0, vmax=1)
    ax.set_title(name, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Kernel Matrices Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Quantum Advantage in Kernel Methods

### 4.1 When Can Quantum Kernels Help?

Huang et al. (2021, *Nature*) showed that quantum advantage in kernel methods requires:

1. **Large geometric difference** $g(K_Q, K_C) \gg 1$ between quantum and best classical kernel
2. **High kernel-target alignment** of the quantum kernel with the prediction task

If condition 1 holds but not condition 2, the quantum kernel is exotic but useless. If condition 2 holds but not condition 1, a classical kernel works just as well.

### 4.2 The Curse of Exponential Concentration

**Problem:** As the number of qubits $n$ grows, quantum kernels tend to **exponentially concentrate**:

$$\kappa(\mathbf{x}, \mathbf{x}') \to \frac{1}{2^n} \quad \text{for } \mathbf{x} \neq \mathbf{x}'$$

This means the kernel matrix approaches the identity matrix — every point looks equally (dis)similar to every other point. The SVM then memorizes the training data without generalizing.

### 4.3 Mitigation: Projected Quantum Kernels

**Projected quantum kernels** (Huang et al., 2021) address this by projecting the quantum state to classical shadows before computing the kernel:

$$\tilde{\kappa}(\mathbf{x}, \mathbf{x}') = \exp\left(-\gamma \sum_i \|\rho_i(\mathbf{x}) - \rho_i(\mathbf{x}')\|_F^2\right)$$

where $\rho_i(\mathbf{x}) = \text{Tr}_{\bar{i}}(|\phi(\mathbf{x})\rangle\langle\phi(\mathbf{x})|)$ is the reduced density matrix of qubit $i$.

This avoids exponential concentration while preserving quantum features.

In [ ]:
# Demonstrate exponential concentration of quantum kernels

def average_offdiag_kernel(feature_map_fn, n_qubits, n_samples=30):
    """Compute average off-diagonal kernel value."""
    dev_k = qml.device('default.qubit', wires=n_qubits)
    
    @qml.qnode(dev_k)
    def k_circuit(x1, x2):
        feature_map_fn(x2, wires=range(n_qubits))
        qml.adjoint(feature_map_fn)(x1, wires=range(n_qubits))
        return qml.probs(wires=range(n_qubits))
    
    # Random data points
    X = np.random.uniform(0, 2*np.pi, size=(n_samples, n_qubits))
    
    kernel_vals = []
    for i in range(n_samples):
        for j in range(i + 1, n_samples):
            probs = k_circuit(X[i], X[j])
            kernel_vals.append(probs[0])
    
    return np.mean(kernel_vals), np.std(kernel_vals)

# Feature map that can work with any number of qubits
def scalable_feature_map(x, wires):
    n = len(wires)
    for i, w in enumerate(wires):
        qml.Hadamard(wires=w)
        qml.RZ(x[i], wires=w)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i + 1]])
        qml.RZ(x[i] * x[(i+1) % n], wires=wires[i + 1])
        qml.CNOT(wires=[wires[i], wires[i + 1]])
    for i, w in enumerate(wires):
        qml.Hadamard(wires=w)
        qml.RZ(x[i], wires=w)

# Compute average kernel values for different qubit counts
qubit_range = [2, 3, 4, 5, 6, 7, 8]
avg_kernels = []
std_kernels = []

np.random.seed(42)
print("Computing average off-diagonal kernel values...")
for nq in qubit_range:
    mean_k, std_k = average_offdiag_kernel(scalable_feature_map, nq, n_samples=20)
    avg_kernels.append(mean_k)
    std_kernels.append(std_k)
    print(f"  n={nq}: mean κ = {mean_k:.6f}, 1/2^n = {1/2**nq:.6f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.errorbar(qubit_range, avg_kernels, yerr=std_kernels, fmt='bo-', linewidth=2, capsize=5)
ax1.plot(qubit_range, [1/2**n for n in qubit_range], 'r--', linewidth=2, label='1/2^n')
ax1.set_xlabel('Number of Qubits', fontsize=12)
ax1.set_ylabel('Average Off-Diagonal Kernel Value', fontsize=12)
ax1.set_title('Kernel Value Concentration', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.semilogy(qubit_range, avg_kernels, 'bo-', linewidth=2, label='Measured')
ax2.semilogy(qubit_range, [1/2**n for n in qubit_range], 'r--', linewidth=2, label='1/2^n')
ax2.set_xlabel('Number of Qubits', fontsize=12)
ax2.set_ylabel('Average Kernel Value (log scale)', fontsize=12)
ax2.set_title('Exponential Concentration', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAs n increases, kernel values → 1/2^n (approaching identity matrix).")
print("This means the kernel becomes less informative for classification.")

## 5. Projected Quantum Kernels

### 5.1 Implementation

The projected quantum kernel uses reduced density matrices:

$$\tilde{\kappa}(\mathbf{x}, \mathbf{x}') = \exp\left(-\gamma \sum_{i=1}^{n} \|\rho_i(\mathbf{x}) - \rho_i(\mathbf{x}')\|_F^2\right)$$

For each qubit $i$, the reduced density matrix is:

$$\rho_i = \text{Tr}_{\bar{i}}(|\phi(\mathbf{x})\rangle\langle\phi(\mathbf{x})|) = \frac{1}{2}\begin{pmatrix} 1 + \langle Z_i \rangle & \langle X_i \rangle - i \langle Y_i \rangle \\ \langle X_i \rangle + i \langle Y_i \rangle & 1 - \langle Z_i \rangle \end{pmatrix}$$

So we need to measure $\langle X_i \rangle$, $\langle Y_i \rangle$, $\langle Z_i \rangle$ for each qubit.

### 5.2 Advantage Over Fidelity Kernel

The projected kernel avoids exponential concentration because:
- It uses **local** (single-qubit) information instead of global overlap
- The Bloch vector components $\langle X_i \rangle, \langle Y_i \rangle, \langle Z_i \rangle$ remain $O(1)$

In [ ]:
# Implement projected quantum kernel

def make_projected_kernel(feature_map_fn, n_qubits, gamma=1.0):
    """Create a projected quantum kernel."""
    dev_pk = qml.device('default.qubit', wires=n_qubits)
    
    @qml.qnode(dev_pk)
    def bloch_circuit(x):
        feature_map_fn(x, wires=range(n_qubits))
        # Return expectation values for all Pauli operators on each qubit
        return [
            qml.expval(qml.PauliX(i)) for i in range(n_qubits)
        ] + [
            qml.expval(qml.PauliY(i)) for i in range(n_qubits)
        ] + [
            qml.expval(qml.PauliZ(i)) for i in range(n_qubits)
        ]
    
    def get_bloch_vectors(x):
        """Get Bloch vectors for all qubits."""
        results = bloch_circuit(x)
        bloch = np.zeros((n_qubits, 3))
        for i in range(n_qubits):
            bloch[i, 0] = results[i]                  # X
            bloch[i, 1] = results[n_qubits + i]       # Y
            bloch[i, 2] = results[2 * n_qubits + i]   # Z
        return bloch
    
    def projected_kernel(x1, x2):
        """Compute projected kernel value."""
        b1 = get_bloch_vectors(x1)
        b2 = get_bloch_vectors(x2)
        # Sum of squared Frobenius norms of density matrix differences
        # ||rho_i(x1) - rho_i(x2)||_F^2 = (1/2)||b_i(x1) - b_i(x2)||^2
        dist_sq = 0.5 * np.sum((b1 - b2)**2)
        return np.exp(-gamma * dist_sq)
    
    return projected_kernel

# Compare projected kernel vs fidelity kernel on concentration
proj_kernel_4q = make_projected_kernel(scalable_feature_map, 4, gamma=1.0)
fid_kernel_4q = make_kernel_circuit(lambda x, wires: scalable_feature_map(x, wires), 4)

# Generate random data
np.random.seed(42)
n_test_points = 15
X_test_proj = np.random.uniform(0, 2*np.pi, size=(n_test_points, 4))

K_fid = compute_kernel_matrix(fid_kernel_4q, X_test_proj)
K_proj = compute_kernel_matrix(proj_kernel_4q, X_test_proj)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im1 = axes[0].imshow(K_fid, cmap='viridis', vmin=0, vmax=1)
axes[0].set_title('Fidelity Kernel (4 qubits)', fontsize=12)
plt.colorbar(im1, ax=axes[0], fraction=0.046)

im2 = axes[1].imshow(K_proj, cmap='viridis', vmin=0, vmax=1)
axes[1].set_title('Projected Kernel (4 qubits)', fontsize=12)
plt.colorbar(im2, ax=axes[1], fraction=0.046)

# Distribution of off-diagonal values
offdiag_fid = K_fid[np.triu_indices(n_test_points, k=1)]
offdiag_proj = K_proj[np.triu_indices(n_test_points, k=1)]

axes[2].hist(offdiag_fid, bins=20, alpha=0.6, label=f'Fidelity (mean={np.mean(offdiag_fid):.3f})')
axes[2].hist(offdiag_proj, bins=20, alpha=0.6, label=f'Projected (mean={np.mean(offdiag_proj):.3f})')
axes[2].set_xlabel('Kernel Value', fontsize=12)
axes[2].set_ylabel('Count', fontsize=12)
axes[2].set_title('Off-Diagonal Value Distribution', fontsize=12)
axes[2].legend(fontsize=10)

plt.tight_layout()
plt.show()

print("The projected kernel has more spread in off-diagonal values,")
print("making it more informative for classification than the fidelity kernel.")

## 6. Quantum Kernel SVM on the Iris Dataset

### 6.1 The Iris Dataset

The classic Iris dataset has:
- 150 samples, 4 features (sepal/petal length and width)
- 3 classes (setosa, versicolor, virginica)

We will use a binary classification (2 classes) for simplicity.

### 6.2 Approach

1. Compute the quantum kernel matrix $K_Q$
2. Use `sklearn.svm.SVC(kernel='precomputed')` with $K_Q$
3. Compare with classical kernels (linear, RBF)

In [ ]:
# Load and prepare the Iris dataset (binary: classes 0 and 1)

iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Select two classes for binary classification
mask = y_iris < 2  # Classes 0 (setosa) and 1 (versicolor)
X_iris_bin = X_iris[mask]
y_iris_bin = y_iris[mask]
y_iris_bin = 2 * y_iris_bin - 1  # Convert to {-1, +1}

# Use only 2 features for visualization (petal length and width)
X_2f = X_iris_bin[:, 2:4]

# Scale to [0, pi]
scaler_iris = MinMaxScaler(feature_range=(0, np.pi))
X_scaled_iris = scaler_iris.fit_transform(X_2f)

# Train/test split
X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_scaled_iris, y_iris_bin, test_size=0.3, random_state=42
)

print(f"Dataset: {len(X_train_i)} training, {len(X_test_i)} test samples")
print(f"Features: petal length and petal width (scaled to [0, π])")
print(f"Classes: setosa (-1) vs versicolor (+1)")

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_scaled_iris[:, 0], X_scaled_iris[:, 1], c=y_iris_bin, 
                     cmap='RdBu', edgecolors='black', s=60, alpha=0.8)
ax.set_xlabel('Petal Length (scaled)', fontsize=12)
ax.set_ylabel('Petal Width (scaled)', fontsize=12)
ax.set_title('Iris Dataset (Binary Classification)', fontsize=14)
plt.colorbar(scatter, label='Class')
plt.tight_layout()
plt.show()

In [ ]:
# Compute quantum kernel matrices for Iris

print("Computing quantum kernel matrices...")

# IQP kernel
kernel_fn_iqp = make_kernel_circuit(feature_map_iqp, 2)
K_train_q = compute_kernel_matrix(kernel_fn_iqp, X_train_i)
K_test_q = compute_kernel_matrix(kernel_fn_iqp, X_test_i, X_train_i)

print(f"Training kernel matrix shape: {K_train_q.shape}")
print(f"Test kernel matrix shape: {K_test_q.shape}")
print(f"Kernel matrix is PSD: {np.all(np.linalg.eigvalsh(K_train_q) >= -1e-10)}")

In [ ]:
# Train and evaluate SVMs with different kernels

results = {}

# 1. Quantum Kernel SVM (IQP)
svm_quantum = SVC(kernel='precomputed', C=1.0)
svm_quantum.fit(K_train_q, y_train_i)
y_pred_q = svm_quantum.predict(K_test_q)
acc_q = accuracy_score(y_test_i, y_pred_q)
results['Quantum (IQP)'] = acc_q

# 2. Classical Linear SVM
svm_lin = SVC(kernel='linear', C=1.0)
svm_lin.fit(X_train_i, y_train_i)
y_pred_lin = svm_lin.predict(X_test_i)
acc_lin = accuracy_score(y_test_i, y_pred_lin)
results['Classical (Linear)'] = acc_lin

# 3. Classical RBF SVM
svm_rbf_iris = SVC(kernel='rbf', gamma='scale', C=1.0)
svm_rbf_iris.fit(X_train_i, y_train_i)
y_pred_rbf = svm_rbf_iris.predict(X_test_i)
acc_rbf = accuracy_score(y_test_i, y_pred_rbf)
results['Classical (RBF)'] = acc_rbf

# 4. Classical Polynomial SVM
svm_poly = SVC(kernel='poly', degree=3, C=1.0)
svm_poly.fit(X_train_i, y_train_i)
y_pred_poly = svm_poly.predict(X_test_i)
acc_poly = accuracy_score(y_test_i, y_pred_poly)
results['Classical (Poly)'] = acc_poly

# Display results
print("\n" + "=" * 50)
print("Classification Results on Iris Dataset")
print("=" * 50)
for name, acc in results.items():
    print(f"{name:25s}: {acc:.2%}")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 6))
names = list(results.keys())
accs = list(results.values())
colors = ['steelblue', 'lightcoral', 'lightgreen', 'wheat']
bars = ax.bar(names, accs, color=colors, edgecolor='black', alpha=0.8)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Quantum vs Classical Kernel SVM on Iris', fontsize=14)
ax.set_ylim(0, 1.1)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{acc:.1%}', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize decision boundaries for quantum vs classical SVM

# Create mesh for decision boundary
h = 0.05
x_min, x_max = 0, np.pi
y_min, y_max = 0, np.pi
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Classical RBF
ax = axes[0]
Z_rbf_iris = svm_rbf_iris.decision_function(grid).reshape(xx.shape)
ax.contourf(xx, yy, Z_rbf_iris, levels=20, cmap='RdBu', alpha=0.5)
ax.contour(xx, yy, Z_rbf_iris, levels=[0], colors='black', linewidths=2)
ax.scatter(X_train_i[:, 0], X_train_i[:, 1], c=y_train_i, cmap='RdBu', 
           edgecolors='black', s=60)
ax.set_title(f'Classical RBF SVM ({acc_rbf:.1%})', fontsize=12)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')

# Classical Linear
ax = axes[1]
Z_lin_iris = svm_lin.decision_function(grid).reshape(xx.shape)
ax.contourf(xx, yy, Z_lin_iris, levels=20, cmap='RdBu', alpha=0.5)
ax.contour(xx, yy, Z_lin_iris, levels=[0], colors='black', linewidths=2)
ax.scatter(X_train_i[:, 0], X_train_i[:, 1], c=y_train_i, cmap='RdBu', 
           edgecolors='black', s=60)
ax.set_title(f'Classical Linear SVM ({acc_lin:.1%})', fontsize=12)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')

# Quantum kernel boundary (need to compute kernel for each grid point)
ax = axes[2]
print("Computing quantum decision boundary (this may take a while)...")
K_grid_q = compute_kernel_matrix(kernel_fn_iqp, grid[::5], X_train_i)  # Subsample grid
Z_q = svm_quantum.decision_function(K_grid_q)

# Interpolate for plotting
from scipy.interpolate import griddata
Z_q_full = griddata(grid[::5], Z_q, grid, method='linear', fill_value=0)
Z_q_grid = Z_q_full.reshape(xx.shape)

ax.contourf(xx, yy, Z_q_grid, levels=20, cmap='RdBu', alpha=0.5)
ax.contour(xx, yy, Z_q_grid, levels=[0], colors='black', linewidths=2)
ax.scatter(X_train_i[:, 0], X_train_i[:, 1], c=y_train_i, cmap='RdBu', 
           edgecolors='black', s=60)
ax.set_title(f'Quantum Kernel SVM ({acc_q:.1%})', fontsize=12)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')

plt.suptitle('Decision Boundaries: Quantum vs Classical SVM', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Multi-class Quantum Kernel SVM

### 7.1 One-vs-Rest Strategy

For $K$-class problems, we train $K$ binary classifiers, each distinguishing one class from all others. `sklearn`'s SVC handles this automatically with `decision_function_shape='ovr'`.

### 7.2 Full Iris Classification

In [ ]:
# Full 3-class Iris classification with quantum kernel

# Use all 3 classes but still 2 features for the quantum kernel
X_3class = iris.data[:, 2:4]  # petal length and width
y_3class = iris.target

scaler_3c = MinMaxScaler(feature_range=(0, np.pi))
X_3c_scaled = scaler_3c.fit_transform(X_3class)

X_train_3c, X_test_3c, y_train_3c, y_test_3c = train_test_split(
    X_3c_scaled, y_3class, test_size=0.3, random_state=42, stratify=y_3class
)

# Compute kernel matrices
print("Computing 3-class quantum kernel matrices...")
K_train_3c = compute_kernel_matrix(kernel_fn_iqp, X_train_3c)
K_test_3c = compute_kernel_matrix(kernel_fn_iqp, X_test_3c, X_train_3c)

# Quantum SVM
svm_q_3c = SVC(kernel='precomputed', C=1.0, decision_function_shape='ovr')
svm_q_3c.fit(K_train_3c, y_train_3c)
y_pred_q_3c = svm_q_3c.predict(K_test_3c)
acc_q_3c = accuracy_score(y_test_3c, y_pred_q_3c)

# Classical SVMs
svm_rbf_3c = SVC(kernel='rbf', gamma='scale', C=1.0)
svm_rbf_3c.fit(X_train_3c, y_train_3c)
acc_rbf_3c = accuracy_score(y_test_3c, svm_rbf_3c.predict(X_test_3c))

svm_lin_3c = SVC(kernel='linear', C=1.0)
svm_lin_3c.fit(X_train_3c, y_train_3c)
acc_lin_3c = accuracy_score(y_test_3c, svm_lin_3c.predict(X_test_3c))

print(f"\n3-Class Iris Classification Results:")
print(f"  Quantum IQP kernel SVM: {acc_q_3c:.2%}")
print(f"  Classical RBF SVM:      {acc_rbf_3c:.2%}")
print(f"  Classical Linear SVM:   {acc_lin_3c:.2%}")

print(f"\nQuantum Kernel SVM Classification Report:")
print(classification_report(y_test_3c, y_pred_q_3c, 
                           target_names=iris.target_names))

## 8. Trainable Quantum Kernels

### 8.1 The Idea

Instead of using a fixed feature map, we can **optimize** the feature map parameters to maximize kernel-target alignment:

$$\boldsymbol{\theta}^* = \arg\max_{\boldsymbol{\theta}} A(K_{\boldsymbol{\theta}}, \mathbf{y})$$

The parameterized feature map becomes:

$$U_{\phi}(\mathbf{x}; \boldsymbol{\theta}) = \prod_l \left[ V_l(\boldsymbol{\theta}_l) \cdot W_l(\mathbf{x}) \right]$$

where $V_l(\boldsymbol{\theta}_l)$ are trainable gates and $W_l(\mathbf{x})$ encode the data.

### 8.2 Training Procedure

1. Compute kernel matrix $K_{\boldsymbol{\theta}}$
2. Compute alignment $A(K_{\boldsymbol{\theta}}, \mathbf{y})$
3. Update $\boldsymbol{\theta}$ via gradient ascent
4. Repeat until convergence
5. Train SVM with optimized kernel

In [ ]:
# Trainable quantum kernel

n_q_train = 2
dev_trainable = qml.device('default.qubit', wires=n_q_train)

@qml.qnode(dev_trainable)
def trainable_kernel_circuit(x1, x2, params):
    """Kernel circuit with trainable parameters."""
    # Encode x2 with trainable feature map
    for i in range(n_q_train):
        qml.RY(params[0, i], wires=i)
        qml.RZ(x2[i], wires=i)
        qml.RY(params[1, i], wires=i)
    qml.CNOT(wires=[0, 1])
    qml.RZ(params[2, 0] * x2[0] * x2[1], wires=1)
    qml.CNOT(wires=[0, 1])
    
    # Adjoint: encode x1
    qml.CNOT(wires=[0, 1])
    qml.RZ(-params[2, 0] * x1[0] * x1[1], wires=1)
    qml.CNOT(wires=[0, 1])
    for i in range(n_q_train - 1, -1, -1):
        qml.RY(-params[1, i], wires=i)
        qml.RZ(-x1[i], wires=i)
        qml.RY(-params[0, i], wires=i)
    
    return qml.probs(wires=range(n_q_train))

def trainable_kernel(x1, x2, params):
    return trainable_kernel_circuit(x1, x2, params)[0]

def trainable_kernel_matrix(X, params):
    n = len(X)
    K = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            K[i, j] = trainable_kernel(X[i], X[j], params)
            K[j, i] = K[i, j]
    return K

def alignment_cost(params, X, y):
    """Negative kernel-target alignment (for minimization)."""
    K = trainable_kernel_matrix(X, params)
    return -kernel_target_alignment(K, y)

# Train the kernel parameters
np.random.seed(42)
params_init = np.random.uniform(-np.pi, np.pi, size=(3, n_q_train))

# Use a small subset for kernel training
X_kernel_train = X_train_i[:25]
y_kernel_train = y_train_i[:25]

print("Training quantum kernel parameters...")
opt = qml.GradientDescentOptimizer(stepsize=0.2)
params = params_init.copy()

alignments = []
for step in range(15):
    params = opt.step(lambda p: alignment_cost(p, X_kernel_train, y_kernel_train), params)
    a = -alignment_cost(params, X_kernel_train, y_kernel_train)
    alignments.append(a)
    if (step + 1) % 5 == 0:
        print(f"  Step {step+1}: Alignment = {a:.4f}")

# Compare initial vs trained alignment
a_init = -alignment_cost(params_init, X_kernel_train, y_kernel_train)
a_final = alignments[-1]
print(f"\nInitial alignment: {a_init:.4f}")
print(f"Final alignment:   {a_final:.4f}")
print(f"Improvement: {(a_final - a_init)/a_init * 100:.1f}%")

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(alignments) + 1), alignments, 'bo-', linewidth=2)
plt.axhline(y=a_init, color='r', linestyle='--', label=f'Initial: {a_init:.4f}')
plt.xlabel('Training Step', fontsize=12)
plt.ylabel('Kernel-Target Alignment', fontsize=12)
plt.title('Trainable Quantum Kernel Optimization', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Computational Complexity Analysis

### 9.1 Cost of Quantum Kernel Methods

| Step | Classical | Quantum |
|------|-----------|--------|
| Kernel matrix computation | $O(N^2 \cdot d)$ | $O(N^2 \cdot M \cdot D)$ |
| SVM training | $O(N^2)$ to $O(N^3)$ | $O(N^2)$ to $O(N^3)$ (classical) |
| Prediction | $O(N_{sv} \cdot d)$ | $O(N_{sv} \cdot M \cdot D)$ |

where $N$ = dataset size, $d$ = feature dimension, $M$ = shots per kernel entry, $D$ = circuit depth, $N_{sv}$ = number of support vectors.

### 9.2 When Is It Worth Using Quantum Kernels?

Quantum kernels are advantageous when:
1. The quantum feature space captures relevant structure that classical features miss
2. The dataset is not too large ($N \lesssim 10^3$, since kernel matrix is $N \times N$)
3. The feature dimension in the quantum Hilbert space ($2^n$) would be intractable classically
4. The geometric difference is provably large

### 9.3 Open Questions

- Can we find **natural** datasets where quantum kernels provably outperform classical?
- How do quantum kernels perform with realistic **noise** and finite **shots**?
- Can **trainable** quantum kernels escape barren plateaus better than variational models?

In [ ]:
# Final comprehensive comparison

print("=" * 60)
print("COMPREHENSIVE COMPARISON: QUANTUM VS CLASSICAL KERNELS")
print("=" * 60)

# Moon dataset comparison
X_moon, y_moon = make_moons(n_samples=100, noise=0.15, random_state=42)
y_moon = 2 * y_moon - 1
scaler_moon = MinMaxScaler(feature_range=(0, np.pi))
X_moon_sc = scaler_moon.fit_transform(X_moon)
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(X_moon_sc, y_moon, test_size=0.3, random_state=42)

# Quantum kernel
print("\nComputing quantum kernels for Moon dataset...")
K_tr_moon = compute_kernel_matrix(kernel_fn_iqp, X_tr_m)
K_te_moon = compute_kernel_matrix(kernel_fn_iqp, X_te_m, X_tr_m)

svm_q_moon = SVC(kernel='precomputed', C=1.0).fit(K_tr_moon, y_tr_m)
svm_rbf_moon = SVC(kernel='rbf', gamma='scale', C=1.0).fit(X_tr_m, y_tr_m)
svm_lin_moon = SVC(kernel='linear', C=1.0).fit(X_tr_m, y_tr_m)

results_moon = {
    'Quantum (IQP)': accuracy_score(y_te_m, svm_q_moon.predict(K_te_moon)),
    'Classical (RBF)': accuracy_score(y_te_m, svm_rbf_moon.predict(X_te_m)),
    'Classical (Linear)': accuracy_score(y_te_m, svm_lin_moon.predict(X_te_m)),
}

print("\n--- Moon Dataset ---")
for name, acc in results_moon.items():
    print(f"  {name:25s}: {acc:.2%}")

print("\n--- Iris Dataset (Binary) ---")
for name, acc in results.items():
    print(f"  {name:25s}: {acc:.2%}")

print("\n--- Iris Dataset (3-class) ---")
print(f"  {'Quantum (IQP)':25s}: {acc_q_3c:.2%}")
print(f"  {'Classical (RBF)':25s}: {acc_rbf_3c:.2%}")
print(f"  {'Classical (Linear)':25s}: {acc_lin_3c:.2%}")

print("\n" + "=" * 60)
print("CONCLUSION:")
print("On these small classical datasets, quantum and classical kernels")
print("perform comparably. Quantum advantage is expected for datasets")
print("with inherently quantum-like structure or where classical kernels")
print("provably fail to capture relevant features.")
print("=" * 60)

## 10. Summary and Key Takeaways

### What We Learned

| Concept | Key Formula / Idea |
|---------|--------------------|
| Classical kernel trick | $k(\mathbf{x}, \mathbf{x}') = \langle\phi(\mathbf{x}), \phi(\mathbf{x}')\rangle$ |
| Quantum kernel | $\kappa(\mathbf{x}, \mathbf{x}') = |\langle\phi(\mathbf{x})|\phi(\mathbf{x}')\rangle|^2$ |
| Kernel computation | Measure $P(|0^n\rangle)$ after $U_\phi^\dagger(\mathbf{x}) U_\phi(\mathbf{x}')$ |
| Mercer's theorem | Valid kernel $\Leftrightarrow$ PSD kernel matrix |
| Kernel-target alignment | $A(K, y) = \langle K, yy^T\rangle_F / (\|K\|_F \|yy^T\|_F)$ |
| Exponential concentration | $\kappa(\mathbf{x}, \mathbf{x}') \to 1/2^n$ as $n$ grows |
| Projected kernel | Uses reduced density matrices to avoid concentration |
| Geometric difference | $g(K_Q, K_C)$: quantifies potential quantum advantage |
| Trainable kernels | Optimize feature map for kernel-target alignment |

### Key Insights

1. Quantum kernels are always valid (PSD) by construction
2. The IQP feature map creates classically hard-to-simulate kernels
3. Exponential concentration is the main challenge for scaling
4. Projected kernels and trainable kernels are promising solutions
5. Quantum advantage requires both geometric difference AND alignment with the task

---

### Series Conclusion

This concludes notebooks 11-15 of the Quantum Computing from Ground Up series. We have covered:
- **Error correction** for fault-tolerant quantum computing
- **Quantum walks** for algorithmic speedups
- **Adiabatic quantum computing** for optimization
- **Quantum machine learning** with variational circuits
- **Quantum kernel methods** for classification

These topics represent the frontier of quantum computing research with active development toward practical applications.

---

*Notebook 15 of the Quantum Computing from Ground Up series.*